#  Модуль 13. Пайплайны и автоматизация

## Подробный конспект

### 13.1. Зачем нужны пайплайны

До этого момента вы выполняли шаги EDA и FE по отдельности: сначала почистили пропуски, потом закодировали категории, затем масштабировали числа, и только потом обучили модель. Это работает в Jupyter Notebook, но создаёт три серьёзные проблемы.

#### Проблема 1: Data Leakage (утечка данных)

Вы уже знаете, что нельзя масштабировать или заполнять пропуски, «подглядывая» в тестовую выборку. Но когда шагов много, легко ошибиться:

In [ ]:
# FALSE НЕПРАВИЛЬНО: утечка на каждом шаге
imputer.fit(X)  # fit на ВСЕХ данных
X_imputed = imputer.transform(X)

encoder.fit(X_imputed)  # fit на ВСЕХ данных
X_encoded = encoder.transform(X_imputed)

scaler.fit(X_encoded)  # fit на ВСЕХ данных
X_scaled = scaler.transform(X_encoded)

X_train, X_test = train_test_split(X_scaled)  # слишком поздно!

К тому моменту, как вы делаете `train_test_split`, тестовые данные уже «просочились» в статистику imputer, encoder и scaler. Модель «знает» о тесте, и оценка качества будет завышена.

#### Проблема 2: Человеческий фактор

В ноутбуке 50 ячеек. Вы запускаете их не всегда сверху вниз, иногда пропускаете, иногда меняете порядок. Результат зависит от того, в какой последовательности выполнялись ячейки. Через неделю вы сами не вспомните, что и в каком порядке делали.

#### Проблема 3: Продакшен

Когда модель готова, её нужно «завернуть» в сервис. Если преобразования разбросаны по 20 ячейкам, разработчикам придётся вручную переписывать всё на Python-скрипт. Это долго и чревато ошибками.

**Решение — Pipeline (конвейер, пайплайн):** единый объект, который объединяет все шаги предобработки и модель. Он гарантирует правильный порядок, защищает от утечки и легко переносится в продакшен.

### 13.2. sklearn.pipeline.Pipeline

`Pipeline` — это цепочка шагов. Каждый шаг — это пара `(имя, объект)`. Объект должен иметь методы `.fit()`, `.transform()` (или `.fit_transform()`), а последний — `.predict()`.

**Правило работы Pipeline:**
- На **обучающих** данных вызывается `.fit_transform()` для всех шагов, кроме последнего. Последний шаг — только `.fit()`.
- На **тестовых** данных вызывается `.transform()` для всех шагов, кроме последнего. Последний шаг — `.predict()`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Создаём пайплайн
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # шаг 1: заполнение пропусков
    ('scaler', StandardScaler()),                    # шаг 2: масштабирование
    ('model', LogisticRegression(max_iter=1000))     # шаг 3: модель
])

# Обучаем: fit_transform на imputer, fit_transform на scaler, fit на модель
pipe.fit(X_train, y_train)

# Предсказываем: transform на imputer, transform на scaler, predict на модели
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

**Почему это безопасно:** `pipe.fit()` видит только `X_train`. `X_test` попадает только в `pipe.predict()`, где к нему применяются уже **обученные** преобразования.

#### Доступ к шагам пайплайна

In [ ]:
# Именованные шаги
print(pipe.named_steps)
# {'imputer': SimpleImputer(...), 'scaler': StandardScaler(...), 'model': LogisticRegression(...)}

# Доступ к конкретному шагу
imputer = pipe.named_steps['imputer']
print(f"Заполненные значения: {imputer.statistics_}")

# Доступ к модели
coef = pipe.named_steps['model'].coef_

### 13.3. ColumnTransformer: разные преобразования для разных столбцов

В реальных данных есть числовые столбцы, категориальные, даты — и к каждому типу нужен свой подход. `ColumnTransformer` позволяет применять разные пайплайны к разным группам столбцов **параллельно**.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Определяем группы столбцов
numeric_features = ['возраст', 'доход', 'стаж']
categorical_features = ['город', 'пол', 'образование']

# Пайплайн для числовых признаков
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Пайплайн для категориальных признаков
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

# Объединяем
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Финальный пайплайн: препроцессинг + модель
clf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)

**Параметр `remainder`:**
- `'drop'` (по умолчанию): столбцы, не попавшие ни в один трансформер, удаляются.
- `'passthrough'`: столбцы, не попавшие ни в один трансформер, оставляются как есть.

In [ ]:
# Если есть столбцы, которые не нужно трогать (например, уже закодированные флаги)
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
], remainder='passthrough')  # остальные столбцы пропускаем через

#### Выбор столбцов по типу (автоматически)

In [ ]:
from sklearn.compose import make_column_selector

# Автоматически выбирает числовые и категориальные столбцы
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, make_column_selector(dtype_include='number')),
    ('cat', categorical_transformer, make_column_selector(dtype_include='object'))
])

### 13.4. Полный пример: от «грязных» данных до модели

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

np.random.seed(42)

# Создаём «грязные» данные
df = pd.DataFrame({
    'возраст': np.concatenate([np.random.normal(35, 10, 95), [np.nan]*5]),
    'доход': np.concatenate([np.random.lognormal(11, 0.5, 90), [np.nan]*10]),
    'город': np.random.choice(['Москва', 'СПб', 'Казань', None], 100),
    'пол': np.random.choice(['М', 'Ж'], 100),
    'таргет': np.random.binomial(1, 0.3, 100)
})

X = df.drop('таргет', axis=1)
y = df['таргет']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === ЧИСЛОВЫЕ ПРИЗНАКИ ===
numeric_features = ['возраст', 'доход']
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p, validate=False)),  # логарифм!
    ('scaler', StandardScaler())
])

# === КАТЕГОРИАЛЬНЫЕ ПРИЗНАКИ ===
categorical_features = ['город', 'пол']
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# === ОБЪЕДИНЯЕМ ===
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

# === ФИНАЛЬНЫЙ ПАЙПЛАЙН ===
model = Pipeline([
    ('prep', preprocessor),
    ('select', SelectKBest(mutual_info_classif, k=5)),  # отбор признаков
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Обучаем
model.fit(X_train, y_train)

# Предсказываем
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

**Что происходит «под капотом» при `fit`:**
1. `ColumnTransformer` применяет `numeric_pipeline` к числовым столбцам и `categorical_pipeline` к категориальным.
2. Результаты склеиваются в одну матрицу.
3. `SelectKBest` считает mutual information на обучающих данных и оставляет топ-5.
4. `RandomForest` обучается на отобранных признаках.

**Что происходит при `predict`:**
1. Те же преобразования, но без `fit` — только `transform`.
2. `SelectKBest` использует уже посчитанные ранги признаков.
3. `RandomForest` предсказывает.

### 13.5. FeatureUnion: параллельное создание признаков

`FeatureUnion` похож на `ColumnTransformer`, но работает не с **разными столбцами**, а с **одними и теми же** данными, применяя разные трансформации параллельно и склеивая результаты.

**Пример:** из текста одновременно извлекаем TF-IDF и базовые статистики (длина, количество слов).

In [ ]:
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer

# Трансформер, который извлекает длину текста
def get_text_length(X):
    return np.array([len(x) for x in X]).reshape(-1, 1)

length_transformer = FunctionTransformer(get_text_length, validate=False)

# Объединяем TF-IDF и длину текста
text_features = FeatureUnion([
    ('tfidf', TfidfVectorizer(max_features=50)),
    ('length', length_transformer)
])

# Используем в пайплайне
text_pipeline = Pipeline([
    ('features', text_features),
    ('clf', LogisticRegression())
])

> **На практике:** `ColumnTransformer` используется чаще, потому что данные обычно уже разбиты на столбцы разных типов. `FeatureUnion` полезен для NLP и сложной инженерии признаков.

### 13.6. Поиск гиперпараметров внутри пайплайна

Вы можете искать лучшие параметры не только для модели, но и для **всех шагов предобработки**.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Пайплайн
pipe = Pipeline([
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

# Сетка параметров
# Формат: 'шаг__параметр'
param_grid = {
    'imputer__strategy': ['mean', 'median'],
    'scaler__with_mean': [True, False],
    'model__C': [0.1, 1, 10],
    'model__penalty': ['l1', 'l2']
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

print(f"Лучшие параметры: {grid.best_params_}")
print(f"Лучший ROC-AUC: {grid.best_score_:.3f}")

# Лучшая модель — это уже обученный пайплайн
best_model = grid.best_estimator_

**Важно:** `GridSearchCV` оборачивает **весь** пайплайн. Это значит, что для каждого фолда кросс-валидации и для каждой комбинации параметров `imputer` и `scaler` обучаются **заново только на трейне**. Утечки нет.

### 13.7. Сохранение и загрузка пайплайнов

Обученный пайплайн — это единый объект, который можно сохранить и передать в продакшен.

#### joblib (рекомендуется для sklearn)

In [ ]:
import joblib

# Сохраняем
joblib.dump(model, 'pipeline.pkl')

# Загружаем
loaded_model = joblib.load('pipeline.pkl')

# Используем
y_pred = loaded_model.predict(X_new)

#### pickle (стандартный Python)

In [ ]:
import pickle

# Сохраняем
with open('pipeline.pkl', 'wb') as f:
    pickle.dump(model, f)

# Загружаем
with open('pipeline.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

**Что сохраняется:**
- Все шаги пайплайна
- Обученные параметры (статистики imputer, веса scaler, коэффициенты модели)
- Настройки (какие столбцы были в ColumnTransformer)

**Что нужно проверять при загрузке:**
- Новые данные должны иметь **те же названия столбцов**, что и при обучении.
- `OneHotEncoder(handle_unknown='ignore')` защитит от новых категорий, но лучше заранее продумать обработку.

### 13.8. Версионирование данных: DVC (краткое введение)

**Проблема:** вы обучили модель, сохранили пайплайн, но через месяц источник данных изменился (новые столбцы, другой формат). Вы не помните, на какой версии данных обучались.

**DVC (Data Version Control)** — инструмент, который работает как Git, но для данных и моделей.

**Основные идеи:**
- Данные хранятся отдельно (S3, Google Drive, локальная папка)
- В Git коммитятся только **метаданные** (хеши файлов, версии)
- `dvc checkout` — переключиться на нужную версию данных

In [ ]:
# Инициализация
dvc init

# Добавить данные
dvc add data.csv

# Зафиксировать версию
git add data.csv.dvc
git commit -m "Version 1.0 of dataset"

# Обновить данные, зафиксировать новую версию
dvc add data.csv
git commit -am "Version 2.0: added new features"

> **Для курса:** знайте, что такой инструмент существует. В реальных проектах без версионирования данных невозможно воспроизвести эксперимент.

### 13.9. Логирование экспериментов: MLflow (краткое введение)

**Проблема:** вы пробуете 20 комбинаций параметров, 5 моделей, 3 способа заполнения пропусков. Через неделю вы не помните, какая комбинация дала лучший результат.

**MLflow** — платформа для отслеживания экспериментов.

**Что логируется:**
- Параметры модели и препроцессинга
- Метрики (ROC-AUC, F1, RMSE)
- Артефакты (сохранённые модели, графики)
- Код и версия данных

In [ ]:
import mlflow
from sklearn.metrics import roc_auc_score

mlflow.set_experiment("eda_course_experiment")

with mlflow.start_run():
    # Логируем параметры
    mlflow.log_param("imputer_strategy", "median")
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    
    # Обучаем
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    
    # Логируем метрики
    mlflow.log_metric("roc_auc", auc)
    
    # Сохраняем модель
    mlflow.sklearn.log_model(model, "model")

После этого в веб-интерфейсе MLflow вы видите таблицу всех запусков с параметрами и метриками.

### 13.10. Практический пример: полный воспроизводимый пайплайн

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib

np.random.seed(42)

# 1. СОЗДАЁМ ДАННЫЕ
df = pd.DataFrame({
    'возраст': np.concatenate([np.random.normal(40, 12, 198), [np.nan]*2]),
    'доход': np.random.lognormal(11, 0.6, 200),
    'город': np.random.choice(['Москва', 'СПб', 'Казань', 'Екатеринбург'], 200),
    'пол': np.random.choice(['М', 'Ж', None], 200),
    'таргет': np.random.binomial(1, 0.25, 200)
})

# 2. РАЗДЕЛЕНИЕ (только один раз, до любой обработки!)
X = df.drop('таргет', axis=1)
y = df['таргет']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. ОПРЕДЕЛЯЕМ ГРУППЫ СТОЛБЦОВ
numeric_features = ['возраст', 'доход']
categorical_features = ['город', 'пол']

# 4. СОЗДАЁМ ПРЕОБРАЗОВАТЕЛИ
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

# 5. ОБЪЕДИНЯЕМ
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# 6. ФИНАЛЬНЫЙ ПАЙПЛАЙН
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, random_state=42))
])

# 7. КРОСС-ВАЛИДАЦИЯ (пайплайн защищает от утечки!)
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='roc_auc')
print(f"CV ROC-AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")

# 8. ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ
pipeline.fit(X_train, y_train)

# 9. ОЦЕНКА НА ТЕСТЕ
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(f"\nTest ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")
print(classification_report(y_test, y_pred))

# 10. СОХРАНЕНИЕ
joblib.dump(pipeline, 'production_pipeline.pkl')
print("\nПайплайн сохранён в 'production_pipeline.pkl'")

# 11. ЗАГРУЗКА И ПРЕДСКАЗАНИЕ НА НОВЫХ ДАННЫХ
loaded_pipe = joblib.load('production_pipeline.pkl')

new_data = pd.DataFrame({
    'возраст': [25, np.nan, 45],
    'доход': [50000, 80000, 120000],
    'город': ['Москва', 'СПб', 'Новый_Город'],  # новая категория — handle_unknown='ignore' спасёт
    'пол': ['М', 'Ж', 'М']
})

predictions = loaded_pipe.predict(new_data)
probabilities = loaded_pipe.predict_proba(new_data)[:, 1]

print(f"\nПредсказания для новых данных: {predictions}")
print(f"Вероятности: {probabilities}")

### 13.11. Чек-лист для самопроверки

Перед переходом к практическим проектам убедитесь, что вы:

- [ ] Понимаете, почему разбросанные по ячейкам преобразования опасны (Data Leakage)
- [ ] Можете создать `Pipeline` из нескольких шагов
- [ ] Знаете, что `fit` на пайплайне вызывает `fit_transform` для промежуточных шагов и `fit` для последнего
- [ ] Знаете, что `predict` на пайплайне вызывает `transform` для промежуточных шагов и `predict` для последнего
- [ ] Умеете использовать `ColumnTransformer` для разных типов столбцов
- [ ] Знаете, что делает параметр `remainder='passthrough'`
- [ ] Можете применять `GridSearchCV` к пайплайну и задавать параметры в формате `шаг__параметр`
- [ ] Умеете сохранять и загружать пайплайн через `joblib`
- [ ] Понимаете, зачем нужно версионирование данных (DVC)
- [ ] Знаете, что такое MLflow и зачем логировать эксперименты
- [ ] Можете построить полный пайплайн от сырых данных до предсказания
- [ ] Понимаете, что `handle_unknown='ignore'` в OneHotEncoder защищает от новых категорий в продакшене

> **Переход к Модулю 14:** Теперь, когда вы владеете всеми инструментами EDA и FE в виде воспроизводимых пайплайнов, пришло время закрепить знания на практике. Следующий модуль содержит четыре полноценных проекта: от классического «Титаника» до анализа временных рядов.